# F6-svd-spectral — Session 2: The Spectral Decomposition

*One class session, roughly 85 minutes. Prerequisite: Session 1 of this
unit (eigenpairs, `np.linalg.eig`, the symmetric promises), plus F3's
outer products and Gram matrices.*

**This session:** the dedicated symmetric eigen-solver
`np.linalg.eigh` and the course's pinned reordering idiom; the
**spectral decomposition** $S = Q\Lambda Q^{\mathsf T}$ — a symmetric
matrix taken apart into perpendicular stretch axes; the same fact as a
sum of rank-1 outer-product atoms $\sum_i \lambda_i q_i q_i^{\mathsf T}$;
the course's **sign-fixing pin** for comparing eigenvectors across
routes; and what $x^{\mathsf T} S x$ — the *energy* of a direction —
means and why Gram matrices never have negative eigenvalues.
Plus a fully worked exam-style constrained-coding example.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np

SEED = 20260804

## 1. Symmetric Matrices Are the Ones You Will Actually Meet

Session 1 ended with the two symmetric promises: all-real eigenvalues,
and an orthonormal basis of eigenvectors.
Before cashing them in, note how often the matrices in your workflow
are symmetric *by construction*:

- **Gram matrices** (F3): $G = WW^{\mathsf T}$ collects all pairwise
  row dot products of a data table $W$; symmetric because the dot
  product commutes.
  Every similarity table you built in F3 is one of these.
- **Symmetrized combinations**: for any square $B$, both
  $B + B^{\mathsf T}$ and $B^{\mathsf T} B$ are symmetric — the
  standard ways symmetric test matrices get built (and how this
  session's demo matrix was built in Session 1 §7).
- **Next session's star**: the bridge matrix $S = WW^{\mathsf T}$ that
  connects the SVD of a *rectangular* $W$ to everything in this
  session.

So a dedicated toolkit for symmetric matrices is not a special-interest
corner — it is the main road.

In [ ]:
rng = np.random.default_rng(SEED)
W0 = rng.normal(0, 1, (5, 3))     # a small data table: 5 rows in 3-D
G = W0 @ W0.T                     # Gram matrix of the rows
print("G shape:", G.shape, " symmetry gap:", np.abs(G - G.T).max())
print("G diagonal (squared row lengths):", np.diagonal(G))

### Checkpoint 1

1. Why is $G_{ij} = G_{ji}$ automatic for a Gram matrix?
   One sentence, naming the F2 fact doing the work.
2. For any square $B$, show that $B + B^{\mathsf T}$ is symmetric by
   computing the $(i, j)$ and $(j, i)$ entries.

## 2. `np.linalg.eigh` and the Reorder Idiom

For symmetric input NumPy provides a specialist:

```python
vals, vecs = np.linalg.eigh(S)    # symmetric (Hermitian) matrices only
```

Advantages over `eig`, in exchange for the symmetry requirement:

- **Real float64 output, guaranteed** — no complex dtype to strip
  (promise 1 is baked into the return type).
- **Orthonormal eigenvector columns, guaranteed** — promise 2, exact
  to machine precision even when eigenvalues repeat.
- **Sorted output** — but **ASCENDING**: smallest eigenvalue first.

That last point collides with our descending convention, so the course
pins one idiom — memorize it as a unit:

```python
vals_asc, vecs_asc = np.linalg.eigh(S)
vals = vals_asc[::-1]          # reverse the values...
vecs = vecs_asc[:, ::-1]       # ...and reverse the COLUMNS to match
```

`[::-1]` is F1's reverse-slice; on `vecs` it must act on the *column*
axis (`[:, ::-1]`), because columns are eigenvectors.
This `eigh`-then-reverse idiom is the standard opening move of nearly
every problem in this unit — worth ten seconds of drilling now.

In [ ]:
rng = np.random.default_rng(SEED)
B = rng.normal(0, 1, (5, 5))
S5 = B + B.T                       # the Session 1 §7 matrix, rebuilt

vals_asc, vecs_asc = np.linalg.eigh(S5)
print("eigh raw (ASCENDING):", vals_asc)

vals = vals_asc[::-1]
vecs = vecs_asc[:, ::-1]
print("after the pinned reorder:", vals)
print("dtype:", vals.dtype)

# same numbers eig produced in Session 1, none of the complex ceremony
resid = max(np.abs(S5 @ vecs[:, i] - vals[i] * vecs[:, i]).max()
            for i in range(5))
print("max eigen-equation residual:", resid)
print("max |Q^T Q - I|:", np.abs(vecs.T @ vecs - np.eye(5)).max())

### Checkpoint 2

1. In the reorder idiom, why is the `vecs` reversal `[:, ::-1]` and not
   `[::-1]`?
   What would `vecs[::-1]` actually reverse, and which check exposes
   the mistake?
2. You need the eigenvector for the *largest* eigenvalue of a symmetric
   $S$.
   Give two correct one-liners: one from the raw ascending output, one
   from the reordered arrays.
3. When should you reach for `eig` instead of `eigh`?

## 3. The Spectral Decomposition $S = Q \Lambda Q^{\mathsf T}$

Stack the orthonormal eigenvectors of a symmetric $S$ as the columns of
$Q$ and put the eigenvalues (descending) on the diagonal of
$\Lambda = \mathrm{diag}(\lambda_1, \dots, \lambda_n)$.
Then the two promises assemble into the central identity of this unit:

$$S = Q \, \Lambda \, Q^{\mathsf T}.$$

**Why it is true** (one paragraph, using only F3 machinery).
Both sides are linear maps, and the columns of $Q$ form a basis, so it
is enough to check the two sides agree on each $q_j$
(F3: a linear map is determined by its action on a basis).
Right side: $Q^{\mathsf T} q_j = e_j$ (dot products of $q_j$ with all
the orthonormal columns — zeros except slot $j$); then
$\Lambda e_j = \lambda_j e_j$; then $Q (\lambda_j e_j) = \lambda_j q_j$
(matrix times $e_j$ is column $j$).
Left side: $S q_j = \lambda_j q_j$ — the eigen-equation.
Same answer. ∎

**How to read it as a machine** (right to left):
$Q^{\mathsf T} x$ re-expresses the input in eigen-coordinates (a
rotation — no lengths change);
$\Lambda$ stretches each coordinate by its own $\lambda_i$;
$Q$ rotates back.
*A symmetric matrix is exactly a stretch along $n$ perpendicular
axes.*
That is the geometric cash value of the two promises, and the picture
to hold for the rest of the unit.

In [ ]:
Lam = np.diag(vals)                 # descending, from Section 2
recon = vecs @ Lam @ vecs.T
print("max |Q Lam Q^T - S|:", np.abs(recon - S5).max())

# The machine view, on one input:
x = np.array([1., 2., 0., -1., 1.])
step1 = vecs.T @ x                  # coordinates in the eigenbasis
step2 = vals * step1                # stretch each coordinate
step3 = vecs @ step2                # rotate back
print("pipeline gap vs S5 @ x:", np.abs(step3 - S5 @ x).max())

### Checkpoint 3

1. Hand-assemble the spectral form: $\lambda = (3, 1)$ with
   $q_1 = (1,1)/\sqrt2$ and $q_2 = (1,-1)/\sqrt2$.
   Compute $Q\Lambda Q^{\mathsf T}$ explicitly and name the matrix you
   get (you have met it before).
2. In the machine reading, why does the step $Q^{\mathsf T} x$ change
   no lengths?
   (Name the F2 fact about matrices with orthonormal columns.)
3. If $S = Q\Lambda Q^{\mathsf T}$, what is the spectral decomposition
   of $S + 2I$?
   (Hint: what does $2I$ do to each eigenvalue? Check via
   $S q_i + 2 q_i$.)

## 4. The Atom View: $S = \sum_i \lambda_i\, q_i q_i^{\mathsf T}$

F3 taught outer products as rank-1 atoms and showed every rank-$r$
matrix is a sum of $r$ of them.
The spectral decomposition hands you a *canonical* atom sum: multiply
$Q \Lambda Q^{\mathsf T}$ out column-by-row and you get

$$S = \sum_{i=1}^{n} \lambda_i \, q_i q_i^{\mathsf T},$$

one rank-1 atom per eigenpair, each built from an eigenvector with
*itself*.
Two readings:

- **Structural:** the number of nonzero $\lambda_i$ is the number of
  genuinely contributing atoms — which is why
  $\operatorname{rank}(S) = \#\{i : \lambda_i \ne 0\}$ for symmetric
  $S$.
- **Actional:** $(q_i q_i^{\mathsf T})\,x = (q_i \cdot x)\, q_i$ — the
  atom extracts $x$'s component along $q_i$ (an F2 projection onto a
  unit vector) and $\lambda_i$ scales it.
  The atom sum literally *is* the machine reading of Section 3, one
  axis at a time.

In [ ]:
atoms = [vals[i] * (vecs[:, i][:, None] * vecs[:, i][None, :])
         for i in range(5)]
S_sum = sum(atoms)
print("max |atom sum - S5|:", np.abs(S_sum - S5).max())

# Rank read on the Gram matrix from Section 1: (5,3) rows -> rank <= 3
lam_G = np.linalg.eigh(G)[0][::-1]
print("Gram eigenvalues (desc):", lam_G)
print("nonzero count = rank:", int((np.abs(lam_G) > 1e-10).sum()))

The Gram matrix of five 3-D rows shows **exactly three** nonzero
eigenvalues — its rank, as F3's row-count bound demands
($\operatorname{rank} \le 3$), now visible on the spectrum.
The two machine-zero eigenvalues are flattened directions: collisions,
in F3's language.

**Partial sums preview.**
Keep only the atoms with the *largest* eigenvalues of $G$ and the sum
is already close — the small-$\lambda$ atoms barely matter:

In [ ]:
lam_G, Q_G = np.linalg.eigh(G)
lam_G, Q_G = lam_G[::-1], Q_G[:, ::-1]
partial = np.zeros_like(G)
for k in range(5):
    partial = partial + lam_G[k] * (Q_G[:, k][:, None] * Q_G[:, k][None, :])
    print(f"atoms kept: {k+1}   max |partial - G| = "
          f"{np.abs(partial - G).max():.6f}")

After three atoms the error is machine zero — the last two atoms have
$\lambda \approx 0$ and contribute nothing.
For this matrix all eigenvalues are $\ge 0$ (Section 6 explains why),
so "largest value" and "largest magnitude" agree; *that* is the setting
where dropping small-$\lambda$ atoms is the optimal move, and Sessions
3–4 build the general version of this story (keep few atoms, lose
little) into the unit's main theorem.
For a symmetric matrix with negative eigenvalues, importance is
$|\lambda_i|$, not $\lambda_i$ — keep that footnote in mind.

### Checkpoint 4

1. Write out $\lambda_1 q_1 q_1^{\mathsf T}$ for
   $\lambda_1 = 3$, $q_1 = (1, 1)/\sqrt2$ as an explicit $2 \times 2$
   matrix.
2. A symmetric $4 \times 4$ matrix has eigenvalues $(6, 2, 0, 0)$.
   What is its rank, and how many atoms does its spectral sum
   *genuinely* need?
3. Using the actional reading, what is
   $(q_1 q_1^{\mathsf T})\,q_2$ for orthonormal $q_1, q_2$, and
   why?

## 5. The Sign Pin: Comparing Eigenvectors Across Routes

Session 1 warned that an eigenvector is really a *direction*: $q$ and
$-q$ are both valid, and every route — hand recipe, `eig`, `eigh`, a
future SVD route — makes its own arbitrary sign choice.
The moment you compare two routes entrywise, you need a convention.

> **Course pin (used verbatim from here through the capstone).**
> Individual-vector cross-route comparisons — allowed only for
> eigenvectors of **distinct, well-separated eigenvalues** — either
> compare absolute values, or **sign-fix first: flip each eigenvector
> so that its largest-$|$entry$|$ component is positive.**
> Degenerate or near-degenerate eigenvalues (repeated $\lambda$, e.g.
> the zero-eigenvalue block of a rank-deficient Gram matrix) get NO
> individual-vector comparison at all: within a repeated eigenvalue the
> eigenvectors are not individually pinned down — any rotation of the
> block is equally valid — so verification must use invariant checks
> instead (reconstruction gaps, eigen-equation residuals, projector
> comparisons — Session 5 pins that trio).

The sign-fix, as a function:

In [ ]:
def signfix(Q):
    # Flip each COLUMN of Q so its largest-|entry| component is positive.
    j = np.abs(Q).argmax(axis=0)              # row of the biggest |entry|
    signs = np.sign(Q[j, np.arange(Q.shape[1])])
    return Q * signs                          # broadcast one sign per column


# Cross-route comparison: eig route vs eigh route on S5
ve, Ve = np.linalg.eig(S5)
ve, Ve = ve.real, Ve.real
order = np.argsort(ve)[::-1]
ve, Ve = ve[order], Ve[:, order]

print("eigenvalue gap between routes:", np.abs(ve - vals).max())
print("raw eigenvector gap        :", np.abs(Ve - vecs).max())
print("after sign-fixing BOTH     :", np.abs(signfix(Ve) - signfix(vecs)).max())

Typical story: eigenvalues agree at machine precision immediately
(numbers have no sign freedom), but the raw eigenvector gap is order
$2$ — some columns came back flipped — and sign-fixing both routes
first collapses the gap to machine precision.
$S_5$'s eigenvalues are distinct and well separated, so the pin's
precondition holds and the entrywise comparison is legitimate.

### Checkpoint 5

1. Why must you sign-fix *both* matrices before comparing, rather than
   just one?
2. The pin picks the largest-$|$entry$|$ component as the sign anchor.
   Why is anchoring on, say, "make the first entry positive" fragile?
   (What if the first entry is $10^{-17}$?)
3. A symmetric matrix has eigenvalues $(4, 4, 1)$.
   Which of its eigenvectors may be compared entrywise across routes
   under the pin, and which may not?

## 6. Energy: What $x^{\mathsf T} S x$ Means

For a symmetric $S$ and a *unit* vector $x$, the number

$$E(x) = x^{\mathsf T} S x = x \cdot (Sx)$$

is called the **energy** of the direction $x$ under $S$.
Feed the spectral decomposition into it: writing $c = Q^{\mathsf T} x$
for $x$'s eigen-coordinates,

$$E(x) = x^{\mathsf T} Q \Lambda Q^{\mathsf T} x
       = c^{\mathsf T} \Lambda c = \sum_i \lambda_i c_i^2,
\qquad \sum_i c_i^2 = \lVert x \rVert^2 = 1$$

(the coordinate vector $c$ is unit length because rotation by
$Q^{\mathsf T}$ preserves lengths).
So the energy of any direction is a **weighted average of the
eigenvalues**, weights $c_i^2 \ge 0$ summing to $1$.
Three immediate consequences:

- $E(q_i) = \lambda_i$: each eigen-direction's energy is its own
  eigenvalue (all weight on one slot).
- $\lambda_n \le E(x) \le \lambda_1$ for every unit $x$: no direction
  beats the top eigenvalue, none dips below the bottom one.
  The top eigenvector is *the* energy-maximizing direction.
- **Gram matrices have non-negative eigenvalues** — the answer promised
  in Session 1.
  For $S = WW^{\mathsf T}$:
  $E(x) = x^{\mathsf T} W W^{\mathsf T} x
  = (W^{\mathsf T} x) \cdot (W^{\mathsf T} x)
  = \lVert W^{\mathsf T} x \rVert^2 \ge 0$,
  and taking $x = q_i$ gives $\lambda_i = E(q_i) \ge 0$. ∎
  (Symmetric matrices with all $\lambda_i \ge 0$ are called
  **positive semidefinite**, PSD — the name is worth knowing; Session 3
  leans on this fact when the bridge identifies Gram eigenvalues with
  *squares*.)

Numerical footnote before the check below: eigenvalues that are exactly
$0$ in exact arithmetic surface as $\pm 10^{-16}$-scale fuzz in
float64, so "non-negative" prints as "$\ge$ a tiny negative number" —
treat $|\lambda| < \text{tol}$ as zero, as always.

In [ ]:
# Energies of random unit directions stay inside [lambda_min, lambda_max]
rng = np.random.default_rng(SEED)
lo, hi = vals.min(), vals.max()
worst_lo, worst_hi = np.inf, -np.inf
for _ in range(1000):
    x = rng.normal(0, 1, 5)
    x = x / np.sqrt((x**2).sum())
    E = x @ S5 @ x
    worst_lo, worst_hi = min(worst_lo, E), max(worst_hi, E)
print(f"eigenvalue range: [{lo:.6f}, {hi:.6f}]")
print(f"energy range over 1000 random unit x: [{worst_lo:.6f}, {worst_hi:.6f}]")

# eigen-directions hit their eigenvalues exactly
print("E(q_0) - lambda_0:", vecs[:, 0] @ S5 @ vecs[:, 0] - vals[0])

# Gram matrices: never negative
print("smallest Gram eigenvalue:", np.linalg.eigh(G)[0].min())

### Checkpoint 6

1. $S$ has spectral data $\lambda = (6, 2, -1)$ with orthonormal
   $q_1, q_2, q_3$, and $x = \tfrac{1}{\sqrt2}(q_1 + q_3)$.
   Compute $E(x)$ from the weighted-average formula, and check it lies
   in $[\lambda_3, \lambda_1]$.
2. Why can no unit direction have energy above $\lambda_1$?
   Argue from the weights $c_i^2$.
3. Is $S_5$ (from Section 2) PSD?
   Answer from its printed spectrum, no computation.

## 7. Worked Exam-Style Example: the Constrained-Coding Register

The exam's coding items name exact identifiers, fix shapes, and scope
the allowed library surface per problem.
Here is a fully worked one in this unit's register — treat it as the
dress rehearsal for practice problems p09 and p14.

---

**Worked exam-style example 2 (constrained coding).**

> Write `spectral_rebuild(S)` for a symmetric matrix `S (n, n)`,
> returning the tuple `(vals, vecs, recon_gap, resid_max)` where
> `vals (n,)` holds the eigenvalues in **descending** order,
> `vecs (n, n)` holds the matching eigenvectors as columns,
> `recon_gap` = max absolute entry of
> $Q\,\mathrm{diag}(\lambda)\,Q^{\mathsf T} - S$, and
> `resid_max` = the largest eigen-equation residual
> $\max_i \max\bigl|S q_i - \lambda_i q_i\bigr|$.
> **Allowed `np.linalg` calls: `np.linalg.eigh` only.**
> Both gaps must be $< 10^{-10}$ on the seeded test matrix.
> Reasoning is not required.

*Solution, spelled out.*

The statement fixes the whole plan: `eigh` (the matrix is symmetric —
and the only legal call), the pinned reorder, a `diag` sandwich for the
reconstruction, and a loop-free residual using broadcasting
(`S @ vecs` computes all $S q_i$ at once; `vecs * vals` scales each
column $q_i$ by $\lambda_i$ — F1 broadcasting doing eigen-work).

In [ ]:
def spectral_rebuild(S):
    vals_asc, vecs_asc = np.linalg.eigh(S)
    vals = vals_asc[::-1]                    # the pinned reorder idiom
    vecs = vecs_asc[:, ::-1]
    recon_gap = np.abs(vecs @ np.diag(vals) @ vecs.T - S).max()
    resid_max = np.abs(S @ vecs - vecs * vals).max()
    return vals, vecs, recon_gap, resid_max


vals7, vecs7, recon_gap, resid_max = spectral_rebuild(S5)
print("vals:", vals7)
print("recon_gap:", recon_gap, " resid_max:", resid_max)
assert recon_gap < 1e-10 and resid_max < 1e-10
print("contract satisfied")

Note the residual one-liner: column $i$ of `S @ vecs` is $S q_i$ and
column $i$ of `vecs * vals` is $\lambda_i q_i$ (row-vector broadcast
scales column $i$ by `vals[i]`), so their difference stacks *all* the
residuals — no loop, no ban risk.

### Checkpoint 7

1. Exactly which line would change, and how, if the statement had
   demanded eigenvalues in *ascending* order but eigenvectors still
   paired correctly?
2. Why does `vecs * vals` scale *columns* by `vals` rather than rows?
   (Think broadcast shapes: `(n, n) * (n,)`.)

## 8. Common Pitfalls II

**Pitfall 1: forgetting the reorder entirely.**
`eigh` output *looks* sorted, so the bug survives casual inspection —
until a "top eigenvector" is actually the bottom one.
Symptom: `vals[0]` is the *most negative* eigenvalue.
Fix: the idiom, every time.

**Pitfall 2: reversing values but not columns (or rows instead of
columns).**
Both leave `vals[i]` paired with the wrong column, and both are caught
instantly by the eigen-equation residual:

In [ ]:
vals_asc, vecs_asc = np.linalg.eigh(S5)

# Correct pairing:
v_ok, Q_ok = vals_asc[::-1], vecs_asc[:, ::-1]
# Bug A: values reversed, columns forgotten
v_a, Q_a = vals_asc[::-1], vecs_asc
# Bug B: ROWS reversed instead of columns
v_b, Q_b = vals_asc[::-1], vecs_asc[::-1]

for name, v, Q in [("correct", v_ok, Q_ok), ("bug A", v_a, Q_a),
                   ("bug B", v_b, Q_b)]:
    resid = np.abs(S5 @ Q - Q * v).max()
    print(f"{name:8s} residual: {resid:.2e}")

**Pitfall 3: `eigh` on a non-symmetric matrix.**
`eigh` does not check symmetry — it silently reads only one triangle of
the input and answers a question about a *different* (symmetrized)
matrix.
On genuinely non-symmetric input the result is simply wrong for your
matrix.
Fix: `eigh` only when symmetry is true by construction (build
$B + B^{\mathsf T}$, $WW^{\mathsf T}$…), or asserted:
`np.abs(S - S.T).max() < tol`.

**Pitfall 4: entrywise-comparing eigenvectors without the pin.**
Two correct routes can disagree on every sign; a naive
`np.abs(Q1 - Q2).max()` then reports order-$2$ "errors" on perfect
computations — or worse, near-degenerate eigenvalues make entrywise
comparison meaningless no matter the signs.
Fix: Section 5's pin — sign-fix both under the distinct-and-separated
precondition, invariants otherwise.

**Pitfall 5: treating `vals` as the $\Lambda$ matrix.**
`vecs @ vals @ vecs.T` raises no error — `vals (n,)` broadcasts as a
vector and produces garbage shapes or silent nonsense depending on
context.
The sandwich needs `np.diag(vals)`; the atom sum and the residual
idiom are the vector-friendly forms.

### Checkpoint 8

1. In the Bug A / Bug B demo, why is the *correct* residual
   $\sim 10^{-15}$ rather than exactly $0$?
2. A teammate calls `np.linalg.eigh(M)` on a matrix with
   `np.abs(M - M.T).max() == 0.8` and reports beautiful small
   residuals… against which matrix are those residuals actually small,
   and which one-line assert would have caught the misuse?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $G_{ij} = w_i \cdot w_j$ and $G_{ji} = w_j \cdot w_i$ — equal
   because the dot product commutes (F2).
2. $(B + B^{\mathsf T})_{ij} = B_{ij} + B_{ji}$ and
   $(B + B^{\mathsf T})_{ji} = B_{ji} + B_{ij}$ — the same sum.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Eigenvectors are the COLUMNS, so the column axis must be reversed;
   `vecs[::-1]` reverses the *rows*, scrambling every eigenvector's
   entries while leaving the column order alone.
   The eigen-equation residual (order-1 instead of $10^{-15}$) exposes
   it.
2. Raw ascending: `vecs_asc[:, -1]` (last column pairs with the largest
   value).
   Reordered: `vecs[:, 0]`.
3. When the matrix is not symmetric — `eigh` would silently answer for
   a different matrix (Pitfall 3); `eig` is the general tool.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $3 \cdot \tfrac12\begin{pmatrix}1&1\\1&1\end{pmatrix}
   + 1 \cdot \tfrac12\begin{pmatrix}1&-1\\-1&1\end{pmatrix}
   = \begin{pmatrix}2&1\\1&2\end{pmatrix}$ — Session 1's fan-demo
   matrix $A$, whose eigen-structure we measured empirically there.
2. $Q^{\mathsf T}$ has orthonormal *rows* (the $q_i^{\mathsf T}$), and
   maps with orthonormal rows/columns preserve dot products and hence
   lengths (F2: rotations).
   Concretely $\lVert Q^{\mathsf T}x \rVert^2 = x^{\mathsf T} Q
   Q^{\mathsf T} x = x^{\mathsf T} x$ here since $Q$ is square
   orthonormal.
3. $S + 2I = Q(\Lambda + 2I)Q^{\mathsf T}$: same eigenvectors, every
   eigenvalue shifted up by $2$ —
   $(S + 2I)q_i = \lambda_i q_i + 2q_i = (\lambda_i + 2)q_i$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $3 \cdot \tfrac12 \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}
   = \begin{pmatrix} 1.5 & 1.5 \\ 1.5 & 1.5 \end{pmatrix}$.
2. Rank 2 (two nonzero eigenvalues); two atoms — the $\lambda = 0$
   atoms are zero matrices.
3. $(q_1 q_1^{\mathsf T})q_2 = (q_1 \cdot q_2)\,q_1 = 0 \cdot q_1 = 0$:
   the atom projects onto $q_1$'s axis, and $q_2$ has no component
   there (orthogonality).

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. The convention only produces a canonical representative if applied
   to *each* side; sign-fixing one side merely re-randomizes which
   columns disagree.
2. An anchor entry near $0$ has an unstable sign (it can flip under
   $10^{-16}$-level noise between routes), flipping the whole column
   with it; the largest-$|$entry$|$ component is as far from the
   sign boundary as the vector allows.
3. The $\lambda = 1$ eigenvector may be compared entrywise (distinct
   and separated); the two $\lambda = 4$ eigenvectors may NOT — inside
   that repeated eigenvalue any rotation of the pair is equally valid,
   so only invariant checks (e.g. their projector) are meaningful.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Weights: $c = (1/\sqrt2, 0, 1/\sqrt2)$, so
   $E = \tfrac12 \cdot 6 + \tfrac12 \cdot (-1) = 2.5$ —
   inside $[-1, 6]$. ✓
2. $\sum_i \lambda_i c_i^2 \le \sum_i \lambda_1 c_i^2
   = \lambda_1 \sum_i c_i^2 = \lambda_1$: replacing every eigenvalue by
   the largest can only increase the weighted average.
3. No: its spectrum contains negative values (e.g. $-6.40$), so it is
   not PSD — consistent with being built as $B + B^{\mathsf T}$ rather
   than as a Gram matrix.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Only the two reorder lines: keep `vals = vals_asc` and
   `vecs = vecs_asc` (no reversal at all) — `eigh` already returns
   ascending order with matched columns.
2. Broadcasting `(n, n) * (n,)` aligns the vector with the LAST axis —
   the column index — so entry $(i, j)$ becomes
   `vecs[i, j] * vals[j]`: column $j$ scaled by `vals[j]`, which is
   exactly $\lambda_j q_j$ stacked for all $j$.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Floating-point roundoff: `eigh`'s eigenpairs are computed to about
   machine precision ($\sim 10^{-16}$ per operation), so residuals sit
   at that scale rather than at exact zero — which is why our checks
   are "$<$ tol", never "$== 0$".
2. Small against the symmetrized matrix `eigh` actually analyzed
   (it read one triangle of `M`), not against `M` itself.
   `assert np.abs(M - M.T).max() < 1e-12` before the call.

</details>